# Utilización de la técnica K-Fold

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score, roc_auc_score

In [ ]:
# -----------------------------
# 1️⃣ Cargar datos
# -----------------------------

train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

target_col = "AdoptionSpeed"
X = train.set_index("PetID").drop(target_col, axis=1).select_dtypes(exclude="O")
y = train[target_col]
X_test = test.set_index("PetID")[X.columns]

In [ ]:
# 2️⃣ Configuración de K-Fold
# -----------------------------

from sklearn.model_selection import StratifiedKFold

# Número de divisiones (folds)
n_splits = 5  # comúnmente 5 o 10

# Crear el objeto StratifiedKFold
cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,       # mezcla los datos antes de dividir
    random_state=42     # para reproducibilidad
)

# Verificación rápida: distribución de clases en cada fold
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    print(f"Fold {fold+1}:")
    print(y_train.value_counts(normalize=True).sort_index(), "\n")

Fold 1:
AdoptionSpeed
0    0.027347
1    0.206103
2    0.269218
3    0.217359
4    0.279973
Name: proportion, dtype: float64 

Fold 2:
AdoptionSpeed
0    0.027347
1    0.206103
2    0.269218
3    0.217442
4    0.279890
Name: proportion, dtype: float64 

Fold 3:
AdoptionSpeed
0    0.027347
1    0.206103
2    0.269301
3    0.217359
4    0.279890
Name: proportion, dtype: float64 

Fold 4:
AdoptionSpeed
0    0.027345
1    0.206086
2    0.269279
3    0.217341
4    0.279950
Name: proportion, dtype: float64 

Fold 5:
AdoptionSpeed
0    0.027345
1    0.206086
2    0.269279
3    0.217341
4    0.279950
Name: proportion, dtype: float64 



In [ ]:
# --- Métricas pensadas para desbalance y multiclase ---
scoring = {
    "bal_acc": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "roc_auc_ovr": make_scorer(roc_auc_score, needs_proba=True, multi_class="ovr")
}

In [ ]:
# --- Pipelines de tres modelos (LogisticRegression, RandomForestClassifier, radientBoostingClassifier) ---
base_steps = [("imputer", SimpleImputer(strategy="median"))]

models = {
    "LogReg": Pipeline(base_steps + [
        ("scaler", StandardScaler(with_mean=False)),   # útil para LR; with_mean=False evita problemas si hay sparse
        ("clf", LogisticRegression(
            max_iter=2000, class_weight="balanced", n_jobs=-1, random_state=42
        ))
    ]),
    "RF": Pipeline(base_steps + [
        ("clf", RandomForestClassifier(
            n_estimators=500, max_depth=None, n_jobs=-1,
            class_weight="balanced_subsample", random_state=42
        ))
    ]),
    "GB": Pipeline(base_steps + [
        ("clf", GradientBoostingClassifier(random_state=42))
    ]),
}

In [ ]:
# --- Evaluación CV y resumen de resultados ---
rows = []
for name, pipe in models.items():
    cv_out = cross_validate(
        pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False
    )
    rows.append({
        "model": name,
        "f1_macro_mean": np.mean(cv_out["test_f1_macro"]),
        "f1_macro_std":  np.std(cv_out["test_f1_macro"]),
        "bal_acc_mean":  np.mean(cv_out["test_bal_acc"]),
        "bal_acc_std":   np.std(cv_out["test_bal_acc"]),
        "roc_auc_ovr_mean": np.mean(cv_out["test_roc_auc_ovr"]),
        "roc_auc_ovr_std":  np.std(cv_out["test_roc_auc_ovr"]),
    })

summary = (pd.DataFrame(rows)
           .sort_values(by=["f1_macro_mean","bal_acc_mean"], ascending=False)
           .reset_index(drop=True))
print(summary)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


    model  f1_macro_mean  f1_macro_std  bal_acc_mean  bal_acc_std  \
0      RF       0.333355      0.012198      0.333340     0.008288   
1      GB       0.305331      0.011412      0.316725     0.010146   
2  LogReg       0.243396      0.008792      0.291744     0.010250   

   roc_auc_ovr_mean  roc_auc_ovr_std  
0               NaN              NaN  
1               NaN              NaN  
2               NaN              NaN  


In [ ]:
best_name = summary.loc[0, "model"]     # toma el primero del ranking
best_model = models[best_name]
best_model.fit(X, y)

test_preds = best_model.predict(X_test)

submission = pd.DataFrame({
    "PetID": X_test.index,
    "AdoptionSpeed": test_preds
})
submission.to_csv("submission.csv", index=False)
print(best_name)
print(submission.head())

RF
       PetID  AdoptionSpeed
0  e2dfc2935              4
1  f153b465f              3
2  3c90f3f54              2
3  e02abc8a3              4
4  09f0df7d1              4
